In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import time
from datetime import datetime

import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.build import molecule
from ase.optimize import BFGS
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.geometry import get_distances
from ase.units import fs
from fairchem.core import pretrained_mlip, FAIRChemCalculator

W0418 16:07:34.996000 5020 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.


In [2]:
# ── Calculator ────────────────────────────────────────────────────────────────
# initialize
predictor = pretrained_mlip.get_predict_unit('uma-s-1p1', device='cpu')
calc = FAIRChemCalculator(predictor, task_name='omat')

In [3]:
# ── Load MOF ──────────────────────────────────────────────────────────────────
mof_74 = read('mg_mof74.cif')

# ── Reference energies — always recomputed with the current calc ──────────────
# FIX: The original if/else reused E_mof/E_co2 from a previous session when
# structures were already in memory. Absolute energies are NOT transferable
# across calculator instances — this caused the +5 eV E_bind drift.
print('Computing E_mof...')
mof_ref = mof_74.copy()
mof_ref.calc = calc
BFGS(mof_ref, logfile='opt_mof_74.log').run(fmax=0.01)
E_mof = mof_ref.get_potential_energy()

print('Computing E_co2...')
co2_ref = molecule('CO2')
co2_ref.set_pbc(False)
co2_ref.center(vacuum=10.0)
co2_ref.calc = calc
BFGS(co2_ref, logfile='opt_co2.log').run(fmax=0.01)
E_co2 = co2_ref.get_potential_energy()

print(f'E(MOF) = {E_mof:.6f} eV')   # expect ~ -1177 eV
print(f'E(CO2) = {E_co2:.6f} eV')   # expect ~ -22.6 eV

Computing E_mof...
Computing E_co2...
E(MOF) = -1177.415308 eV
E(CO2) = -22.596883 eV


In [4]:
# ── Placement helpers ─────────────────────────────────────────────────────────
def is_valid_position_full(mof, co2, cutoff=1.6):
    dmat = get_distances(mof.positions, co2.positions,
                         cell=mof.get_cell(), pbc=mof.get_pbc())[1]
    return np.min(dmat) > cutoff

def get_directions():
    dirs = [[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]]
    return [np.array(d)/np.linalg.norm(d) for d in dirs]

def place_co2_oriented(mof, co2, metal_index, cutoff=1.6, d_range=(2.0, 4.0), n_dist=20):
    metal_pos = mof.positions[metal_index]
    for direction in get_directions():
        for d_mg_o in np.linspace(d_range[0], d_range[1], n_dist):
            co2_copy = co2.copy()
            co2_axis = co2_copy.positions[2] - co2_copy.positions[0]
            co2_axis /= np.linalg.norm(co2_axis)
            v     = np.cross(co2_axis, direction)
            angle = np.degrees(np.arccos(np.clip(np.dot(co2_axis, direction), -1, 1)))
            if np.linalg.norm(v) > 1e-8:
                co2_copy.rotate(angle, v, center='COM')
            co2_copy.positions += (
                (metal_pos + d_mg_o * direction) - co2_copy.positions[0]
            )
            if is_valid_position_full(mof, co2_copy, cutoff=cutoff):
                return co2_copy, {
                    'metal_index'  : metal_index,
                    'direction'    : direction,
                    'mg_o_distance': d_mg_o
                }
    return None, None

In [5]:
# ── Find best Mg site ─────────────────────────────────────────────────────────
mg_indices = [i for i, a in enumerate(mof_74) if a.symbol == 'Mg']
print(f'Mg indices: {mg_indices}')

best_mg_idx   = 4      # update from your binding energy scan
static_E_bind = -0.57  # eV


Mg indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [6]:
# ── Place CO2 ─────────────────────────────────────────────────────────────────
# ── Place CO2 ─────────────────────────────────────────────────────────────────
co2_placed, meta = place_co2_oriented(mof_74, molecule('CO2'), best_mg_idx)

if co2_placed is None:
    print(f'Site {best_mg_idx} failed, trying all Mg sites...')
    for idx in mg_indices:
        co2_placed, meta = place_co2_oriented(mof_74, molecule('CO2'), idx)
        if co2_placed is not None:
            best_mg_idx = idx
            print(f'Placement succeeded at site {best_mg_idx}')
            print(f'  direction     = {meta["direction"]}')
            print(f'  Mg-O distance = {meta["mg_o_distance"]:.2f} Å')
            break
        else:
            print(f'  Site {idx}: failed')

if co2_placed is None:
    raise RuntimeError('Placement failed at all Mg sites.')
else:
    print(f'\nCO2 placed at Mg site {best_mg_idx}')


CO2 placed at Mg site 4


In [7]:
# ── Build combined system ─────────────────────────────────────────────────────
system_md = mof_74.copy()
system_md += co2_placed
system_md.set_pbc(True)
system_md.calc = calc
print(f'System: {len(system_md)} atoms ({len(mof_74)} MOF + 3 CO2)')

System: 165 atoms (162 MOF + 3 CO2)


In [8]:
# ── Pre-relax ─────────────────────────────────────────────────────────────────
print('Pre-relaxing...')
t0 = time.time()
BFGS(system_md, logfile='md_prerelax.log').run(fmax=0.05)
print(f'Done in {time.time()-t0:.1f} s')

E_bind_prerelax = system_md.get_potential_energy() - E_mof - E_co2
print(f'E_bind (pre-MD) = {E_bind_prerelax:.4f} eV')  # must be negative
if E_bind_prerelax > 0:
    raise ValueError('E_bind positive before MD — check placement.')

Pre-relaxing...
Done in 106.1 s
E_bind (pre-MD) = -0.4115 eV


In [9]:
# ── MD runner ─────────────────────────────────────────────────────────────────
def run_md(system_in, temperature_K, n_steps=1000, timestep_fs=0.5,
           tdamp_fs=100, traj_every=10, log_every=50):

    atoms = system_in.copy()
    atoms.calc = calc

    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
    Stationary(atoms)

    ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
    traj_file = f'md_site{best_mg_idx}_{temperature_K}K_{ts}.traj'
    log_file  = f'md_site{best_mg_idx}_{temperature_K}K_{ts}.log'

    dyn = NoseHooverChainNVT(
        atoms,
        timestep      = timestep_fs * fs,
        temperature_K = temperature_K,
        tdamp         = tdamp_fs * fs,
        trajectory    = traj_file,
        logfile       = log_file,
    )

    records = []

    def log_step():
        step  = dyn.get_number_of_steps()
        E_pot = atoms.get_potential_energy()
        E_b   = E_pot - E_mof - E_co2
        records.append((step, E_pot, E_b))
        if step % log_every == 0:
            print(f'  step {step:4d} | T = {atoms.get_temperature():6.1f} K | '
                  f'E_pot = {E_pot:.4f} eV | E_bind = {E_b:.4f} eV')

    dyn.attach(log_step, interval=traj_every)

    print(f'\n{"="*55}')
    print(f'  MD  |  T = {temperature_K} K  |  {n_steps} steps  |  {len(atoms)} atoms')
    print(f'{"="*55}')

    t_start = time.time()
    dyn.run(steps=n_steps)
    elapsed = time.time() - t_start

    print(f'\n  Done in {elapsed:.1f} s  ({elapsed/n_steps*1000:.0f} ms/step)')
    print(f'  Trajectory saved to {traj_file}')

    return atoms, records, traj_file

In [11]:
# ── Temperature sweep ─────────────────────────────────────────────────────────
temperatures = range(900, 1050, 50)

all_records   = {}
all_trajs     = {}
final_structs = {}

for T in temperatures:
    print(f'\nStarting MD at {T} K...')
    final_T, records_T, traj_T = run_md(
        system_md,            # always from pre-relaxed structure
        temperature_K=T
    )
    all_records[T]   = records_T
    all_trajs[T]     = traj_T
    final_structs[T] = final_T

print('\nAll temperatures complete!')


Starting MD at 900 K...

  MD  |  T = 900 K  |  1000 steps  |  165 atoms
  step    0 | T =  965.4 K | E_pot = -1200.4237 eV | E_bind = -0.4115 eV
  step   50 | T =  522.1 K | E_pot = -1190.7296 eV | E_bind = 9.2826 eV
  step  100 | T =  429.3 K | E_pot = -1188.0198 eV | E_bind = 11.9924 eV
  step  150 | T =  524.9 K | E_pot = -1189.2191 eV | E_bind = 10.7931 eV
  step  200 | T =  488.6 K | E_pot = -1187.7021 eV | E_bind = 12.3101 eV
  step  250 | T =  523.7 K | E_pot = -1187.5793 eV | E_bind = 12.4329 eV
  step  300 | T =  712.4 K | E_pot = -1190.5590 eV | E_bind = 9.4531 eV
  step  350 | T =  621.5 K | E_pot = -1187.5053 eV | E_bind = 12.5069 eV
  step  400 | T =  589.6 K | E_pot = -1185.6959 eV | E_bind = 14.3163 eV
  step  450 | T =  716.6 K | E_pot = -1187.2218 eV | E_bind = 12.7904 eV
  step  500 | T =  703.5 K | E_pot = -1185.7328 eV | E_bind = 14.2794 eV
  step  550 | T =  710.5 K | E_pot = -1184.6312 eV | E_bind = 15.3810 eV
  step  600 | T =  784.4 K | E_pot = -1184.9537 eV |

In [ ]:



# ── Summary statistics ────────────────────────────────────────────────────────
for T, records in sorted(all_records.items()):
    E_binds = np.array([r[2] for r in records])
    eq_cut  = len(E_binds) // 5
    prod    = E_binds[eq_cut:]
    print(f'\nT = {T} K  (production = {len(prod)} frames)')
    print(f'  <E_bind> = {prod.mean():.4f} eV')
    print(f'  std      = {prod.std():.4f} eV')
    print(f'  min/max  = {prod.min():.4f} / {prod.max():.4f} eV')